# 1 Setup

In [1]:
# 1a. Install packages
!pip install dowhy econml scikit-learn pandas numpy openpyxl -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.1/403.1 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.9/245.9 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.3/155.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 72.0 MB/s eta 0:00:00


In [2]:
# 1b. Import libraries
import re
import math
import zipfile
import urllib.request

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, f1_score, recall_score
from econml.dml import CausalForestDML

np.random.seed(42)


# 2 Helper Function

In [3]:
# 2a. Function: fit one CausalForestDML model
def fit_causal_forest(X, T, Y, min_samples_leaf, class_weight=None):
    """Fit one CausalForestDML model with the given settings."""
    model = CausalForestDML(
        model_y=LogisticRegression(class_weight=class_weight),
        model_t=RandomForestClassifier(),
        discrete_outcome=True,
        discrete_treatment=True,
        n_estimators=500,
        min_samples_leaf=min_samples_leaf,
        max_depth=8,
        honest=True,
        random_state=42
    )
    model.fit(Y, T, X=X)
    return model


In [4]:
# 2b. Function: compute all six metrics for one model
def evaluate_model(model, X_test, Y_test, T_test, T0=0, T1=1):
    """Compute CATE CI width, prediction metrics, and fairness metrics on a test set."""
    lower, upper = model.effect_interval(X_test, T0=T0, T1=T1, alpha=0.05)
    ci_width = (upper - lower).mean()

    probs = model.models_y[0][0].predict_proba(X_test)[:, 1]
    preds = (probs >= 0.5).astype(int)

    auc_pr = average_precision_score(Y_test, probs)
    macro_f1 = f1_score(Y_test, preds, average="macro")
    recall = recall_score(Y_test, preds, pos_label=1)

    group0 = (T_test == T0)
    group1 = (T_test == T1)

    tpr0 = preds[group0][Y_test[group0] == 1].mean() if (Y_test[group0] == 1).sum() > 0 else np.nan
    tpr1 = preds[group1][Y_test[group1] == 1].mean() if (Y_test[group1] == 1).sum() > 0 else np.nan
    equal_opportunity_diff = tpr1 - tpr0

    fpr0 = preds[group0][Y_test[group0] == 0].mean() if (Y_test[group0] == 0).sum() > 0 else np.nan
    fpr1 = preds[group1][Y_test[group1] == 0].mean() if (Y_test[group1] == 0).sum() > 0 else np.nan
    equalized_odds_diff = max(abs(tpr1 - tpr0), abs(fpr1 - fpr0))

    return {
        "CI width": round(ci_width, 4),
        "AUC-PR": round(auc_pr, 4),
        "Macro F1": round(macro_f1, 4),
        "At-risk recall": round(recall, 4),
        "Equal Opportunity diff": round(equal_opportunity_diff, 4),
        "Equalized Odds diff": round(equalized_odds_diff, 4),
    }


In [5]:
# 2c. Function: fit baseline + engineered, print comparison table
def run_comparison(X_train, T_train, Y_train, X_test, T_test, Y_test,
                    quarter_weight, label, comparisons=((0, 1),)):
    """Fit baseline and engineered models once, then print a results table
    for each requested (T0, T1) treatment comparison."""
    baseline_model = fit_causal_forest(X_train, T_train, Y_train, min_samples_leaf=5)
    engineered_model = fit_causal_forest(X_train, T_train, Y_train, min_samples_leaf=8,
                                          class_weight=quarter_weight)

    for T0, T1 in comparisons:
        baseline_scores = evaluate_model(baseline_model, X_test, Y_test, T_test, T0, T1)
        engineered_scores = evaluate_model(engineered_model, X_test, Y_test, T_test, T0, T1)

        print(f"--- {label} (treatment {T1} vs {T0}) ---")
        table = pd.DataFrame({"Baseline": baseline_scores, "Engineered": engineered_scores})
        print(table)
        print()

    return baseline_model, engineered_model


# 3 OULAD Data

In [6]:
# 3a. Download and load OULAD
url = "https://archive.ics.uci.edu/static/public/349/open+university+learning+analytics+dataset.zip"
urllib.request.urlretrieve(url, "oulad.zip")
with zipfile.ZipFile("oulad.zip", "r") as zip_ref:
    zip_ref.extractall("oulad_data")

student_info = pd.read_csv("oulad_data/studentInfo.csv")
student_vle = pd.read_csv("oulad_data/studentVle.csv")
student_assessment = pd.read_csv("oulad_data/studentAssessment.csv")

print("Students loaded:", len(student_info))


Students loaded: 32593


In [7]:
# 3b. Build the outcome and the two treatments
student_info["at_risk"] = student_info["final_result"].apply(
    lambda x: 0 if x in ["Pass", "Distinction"] else 1
)

# Treatment 1: gender (1 = Male, 0 = Female)
student_info["gender_num"] = student_info["gender"].apply(lambda x: 1 if x == "M" else 0)

# Treatment 2: location, grouped from the deprivation band (imd_band) into 3 levels
student_info["imd_band"] = student_info["imd_band"].replace("?", pd.NA)
student_info["imd_band"] = student_info["imd_band"].fillna(student_info["imd_band"].mode()[0])

def imd_lower_bound(band_text):
    match = re.search(r"\d+", str(band_text))
    return int(match.group())

student_info["imd_lower_bound"] = student_info["imd_band"].apply(imd_lower_bound)

def group_deprivation(lower_bound):
    if lower_bound <= 20:
        return 0   # high deprivation
    elif lower_bound <= 60:
        return 1   # medium deprivation
    else:
        return 2   # low deprivation

student_info["location_num"] = student_info["imd_lower_bound"].apply(group_deprivation)


In [8]:
# 3c. Build the confounders: prior education, age, engagement, submissions
def encode_education(level):
    levels = {"No Formal quals": 0, "Lower Than A Level": 1, "A Level or Equivalent": 2,
              "HE Qualification": 3, "Post Graduate Qualification": 4}
    return levels.get(level, -1)

student_info["education_num"] = student_info["highest_education"].apply(encode_education)

def encode_age(band):
    bands = {"0-35": 0, "35-55": 1, "55<=": 2}
    return bands.get(band, -1)

student_info["age_num"] = student_info["age_band"].apply(encode_age)

# Engagement proxy: total VLE clicks and number of active days
engagement = student_vle.groupby("id_student").agg(
    total_clicks=("sum_click", "sum"),
    active_days=("date", "nunique")
).reset_index()

# Homework-submission proxy: number of assessments submitted
submitted = student_assessment.groupby("id_student").size().reset_index(name="submitted_count")

oulad_data = student_info.merge(engagement, on="id_student", how="left")
oulad_data = oulad_data.merge(submitted, on="id_student", how="left")
oulad_data["total_clicks"] = oulad_data["total_clicks"].fillna(0)
oulad_data["active_days"] = oulad_data["active_days"].fillna(0)
oulad_data["submitted_count"] = oulad_data["submitted_count"].fillna(0)

print("Any unmatched education or age values?",
      (oulad_data["education_num"] == -1).sum(), (oulad_data["age_num"] == -1).sum())


Any unmatched education or age values? 0 0


In [9]:
# 3d. Scale confounders and set the class weight
oulad_confounders = ["education_num", "age_num", "total_clicks", "active_days", "submitted_count"]

scaler = StandardScaler()
X_oulad = scaler.fit_transform(oulad_data[oulad_confounders])

T_oulad_gender = oulad_data["gender_num"].values
T_oulad_location = oulad_data["location_num"].values
Y_oulad = oulad_data["at_risk"].values

# Class weight for the engineered model: quarter-strength correction of the imbalance ratio
at_risk_n = (Y_oulad == 1).sum()
not_at_risk_n = (Y_oulad == 0).sum()
oulad_ratio = max(at_risk_n, not_at_risk_n) / min(at_risk_n, not_at_risk_n)
oulad_quarter_weight = {0: 1, 1: 1 + (oulad_ratio - 1) * 0.25}

print("OULAD imbalance ratio:", round(oulad_ratio, 2))
print("OULAD quarter weight:", oulad_quarter_weight)


OULAD imbalance ratio: 1.12
OULAD quarter weight: {0: 1, 1: np.float64(1.0296230094247645)}


# 4 OULAD Results

In [22]:
# 4a. Gender: baseline vs engineered
X_train, X_test, T_train, T_test, Y_train, Y_test = train_test_split(
    X_oulad, T_oulad_gender, Y_oulad, test_size=0.2, random_state=42, stratify=Y_oulad
)

run_comparison(X_train, T_train, Y_train, X_test, T_test, Y_test,
               oulad_quarter_weight, label="OULAD - Gender");


--- OULAD - Gender (treatment 1 vs 0) ---
                        Baseline  Engineered
CI width                  0.1078      0.1060
AUC-PR                    0.9288      0.9288
Macro F1                  0.8190      0.8167
At-risk recall            0.8164      0.8202
Equal Opportunity diff   -0.0210     -0.0226
Equalized Odds diff       0.0601      0.0635



In [23]:
# 4b. Location: baseline vs engineered
X_train, X_test, T_train, T_test, Y_train, Y_test = train_test_split(
    X_oulad, T_oulad_location, Y_oulad, test_size=0.2, random_state=42, stratify=Y_oulad
)

run_comparison(X_train, T_train, Y_train, X_test, T_test, Y_test,
               oulad_quarter_weight, label="OULAD - Location",
               comparisons=((0, 1), (0, 2)));


--- OULAD - Location (treatment 1 vs 0) ---
                        Baseline  Engineered
CI width                  0.0917      0.0875
AUC-PR                    0.9289      0.9289
Macro F1                  0.8207      0.8188
At-risk recall            0.8161      0.8181
Equal Opportunity diff   -0.0214     -0.0252
Equalized Odds diff       0.0214      0.0252

--- OULAD - Location (treatment 2 vs 0) ---
                        Baseline  Engineered
CI width                  0.0977      0.0959
AUC-PR                    0.9289      0.9289
Macro F1                  0.8207      0.8188
At-risk recall            0.8161      0.8181
Equal Opportunity diff   -0.0329     -0.0340
Equalized Odds diff       0.0329      0.0340



# 5 GhEduData

In [12]:
# 5a. Load the anonymized GhEduData file
ghedudata = pd.read_excel("GhEduData_Merged_Anonymized.xlsx", sheet_name="GhEduData_Merged")
print("Students loaded:", len(ghedudata))


Students loaded: 995


In [13]:
# 5b. Clean checkbox artifacts left over from the original data collection form
def clean_text(value):
    value = str(value).replace("\u25a1", "").strip()
    return " ".join(value.split())

messy_columns = ["Governance_Type", "Location_Category", "Avg_Class_Size", "Pct_Teachers_Degree",
                  "Teacher_Student_Ratio", "Classroom_Condition", "English_Textbook_Access",
                  "Math_Textbook_Access", "Home_Study_Access", "Socioeconomic_Profile"]

for col in messy_columns:
    ghedudata[col] = ghedudata[col].apply(clean_text)


In [14]:
# 5c. Build the outcome and the three treatments
ghedudata["at_risk"] = (ghedudata["Aggregate"] >= 21).astype(int)
ghedudata["gender_num"] = ghedudata["Gender"].apply(lambda x: 1 if x == "Male" else 0)

def encode_location(value):
    levels = {"Rural": 0, "Peri-urban": 1, "Urban": 2}
    return levels.get(value, -1)

ghedudata["location_num"] = ghedudata["Location_Category"].apply(encode_location)
ghedudata["school_type_num"] = ghedudata["Governance_Type"].apply(lambda x: 1 if "Private" in x else 0)

print("At-risk count:", ghedudata["at_risk"].sum(), "out of", len(ghedudata))


At-risk count: 249 out of 995


In [15]:
# 5d. Build the student-level confounders (used for every treatment)
def encode_attendance(v):
    if "Less than 50%" in v: return 0
    if "50" in v and "74" in v: return 1
    if "75" in v and "90" in v: return 2
    if "More than 90%" in v: return 3
    return -1

def encode_homework(v):
    if "Rarely" in v: return 0
    if "Sometimes" in v: return 1
    if "Often" in v: return 2
    if "Almost always" in v: return 3
    return -1

def encode_participation(v):
    if "Passive" in v: return 0
    if "Very active" in v: return 3
    if "Moderate" in v: return 1
    if "Active" in v: return 2
    return -1

ghedudata["attendance_num"] = ghedudata["Attendance Rate"].apply(encode_attendance)
ghedudata["homework_num"] = ghedudata["Homework Submission Rate"].apply(encode_homework)
ghedudata["participation_num"] = ghedudata["Classroom Participation"].apply(encode_participation)

student_level_confounders = ["Age", "attendance_num", "homework_num", "participation_num"]


In [16]:
# 5e. Build the school-level confounders (used only for gender - see note below)
def encode_class_size(v):
    sizes = {"Fewer than 20": 0, "20\u201335": 1, "36\u201350": 2,
             "51 - 65": 3, "66 - 80": 4, "More than 80": 5}
    return sizes.get(v, -1)

def encode_teacher_qual(v):
    quals = {"Less than 25%": 0, "50\u201375%": 1, "More than 75%": 2}
    return quals.get(v, -1)

def encode_teacher_ratio(v):
    if "fewer than 25" in v: return 0
    if "25\u201335" in v: return 1
    if "36\u201350" in v: return 2
    if "51 - 65" in v: return 3
    if "66 - 80" in v: return 4
    return -1

def encode_condition(v):
    if "Some classrooms are inadequate" in v: return 0
    if "Mostly adequate" in v: return 1
    if "All classrooms adequate" in v: return 2
    return -1

def encode_textbook(v):
    if "3 or more" in v: return 0
    if "between 2" in v: return 1
    if "own copy" in v: return 2
    return -1

def encode_home_study(v):
    if "Fewer than 25%" in v: return 0
    if "25\u201349%" in v: return 1
    if "More than 75%" in v: return 3
    if "50" in v: return 2
    return -1

def encode_socioeconomic(v):
    if "subsistence" in v: return 0
    if "middle income" in v: return 2
    if "high income" in v: return 3
    if "low income" in v: return 1
    return -1

ghedudata["class_size_num"] = ghedudata["Avg_Class_Size"].apply(encode_class_size)
ghedudata["teacher_qual_num"] = ghedudata["Pct_Teachers_Degree"].apply(encode_teacher_qual)
ghedudata["teacher_ratio_num"] = ghedudata["Teacher_Student_Ratio"].apply(encode_teacher_ratio)
ghedudata["classroom_condition_num"] = ghedudata["Classroom_Condition"].apply(encode_condition)
ghedudata["english_textbook_num"] = ghedudata["English_Textbook_Access"].apply(encode_textbook)
ghedudata["math_textbook_num"] = ghedudata["Math_Textbook_Access"].apply(encode_textbook)
ghedudata["home_study_num"] = ghedudata["Home_Study_Access"].apply(encode_home_study)
ghedudata["socioeconomic_num"] = ghedudata["Socioeconomic_Profile"].apply(encode_socioeconomic)

school_level_confounders = ["class_size_num", "teacher_qual_num", "teacher_ratio_num",
                             "classroom_condition_num", "english_textbook_num",
                             "math_textbook_num", "home_study_num", "socioeconomic_num"]

full_confounders = student_level_confounders + school_level_confounders

# Check every encoding matched - every count below should be 0
for col in ["attendance_num", "homework_num", "participation_num"] + school_level_confounders:
    unmatched = (ghedudata[col] == -1).sum()
    if unmatched > 0:
        print("WARNING - unmatched values in", col, ":", unmatched)
print("Encoding check complete.")


Encoding check complete.


In [17]:
# 5f. Scale confounders and set the class weight
scaler_full = StandardScaler()
X_gh_full = scaler_full.fit_transform(ghedudata[full_confounders])

scaler_student = StandardScaler()
X_gh_student = scaler_student.fit_transform(ghedudata[student_level_confounders])

Y_gh = ghedudata["at_risk"].values
T_gh_gender = ghedudata["gender_num"].values
T_gh_location = ghedudata["location_num"].values
T_gh_schooltype = ghedudata["school_type_num"].values

at_risk_n = (Y_gh == 1).sum()
not_at_risk_n = (Y_gh == 0).sum()
gh_ratio = max(at_risk_n, not_at_risk_n) / min(at_risk_n, not_at_risk_n)
gh_quarter_weight = {0: 1, 1: 1 + (gh_ratio - 1) * 0.25}

print("GhEduData imbalance ratio:", round(gh_ratio, 2))
print("GhEduData quarter weight:", gh_quarter_weight)


GhEduData imbalance ratio: 3.0
GhEduData quarter weight: {0: 1, 1: np.float64(1.498995983935743)}


In [18]:
# 5g. Trim students for the school-type overlap problem
# Students whose school type can be predicted with near certainty from the
# confounders are excluded, to satisfy the causal overlap assumption
overlap_check = LogisticRegression(max_iter=1000)
overlap_check.fit(X_gh_student, T_gh_schooltype)
predicted_prob = overlap_check.predict_proba(X_gh_student)[:, 1]

keep_mask = (predicted_prob >= 0.05) & (predicted_prob <= 0.95)

X_gh_schooltype_trimmed = X_gh_student[keep_mask]
T_gh_schooltype_trimmed = T_gh_schooltype[keep_mask]
Y_gh_schooltype_trimmed = Y_gh[keep_mask]

print("Students kept for school type:", keep_mask.sum(), "out of", len(keep_mask))


Students kept for school type: 925 out of 995


# 6 GhEduData Results

In [24]:
# 6a. Gender: baseline vs engineered
X_train, X_test, T_train, T_test, Y_train, Y_test = train_test_split(
    X_gh_full, T_gh_gender, Y_gh, test_size=0.2, random_state=42, stratify=Y_gh
)

run_comparison(X_train, T_train, Y_train, X_test, T_test, Y_test,
               gh_quarter_weight, label="GhEduData - Gender");


--- GhEduData - Gender (treatment 1 vs 0) ---
                        Baseline  Engineered
CI width                  0.2250      0.1898
AUC-PR                    0.6569      0.6618
Macro F1                  0.7232      0.7266
At-risk recall            0.4800      0.6200
Equal Opportunity diff   -0.2531     -0.2193
Equalized Odds diff       0.2531      0.2193



In [25]:
# 6b. Location: baseline vs engineered
X_train, X_test, T_train, T_test, Y_train, Y_test = train_test_split(
    X_gh_student, T_gh_location, Y_gh, test_size=0.2, random_state=42, stratify=Y_gh
)

run_comparison(X_train, T_train, Y_train, X_test, T_test, Y_test,
               gh_quarter_weight, label="GhEduData - Location",
               comparisons=((0, 1), (0, 2)));


--- GhEduData - Location (treatment 1 vs 0) ---
                        Baseline  Engineered
CI width                  0.3652      0.3028
AUC-PR                    0.5534      0.5555
Macro F1                  0.6118      0.6791
At-risk recall            0.3200      0.5400
Equal Opportunity diff   -0.0507      0.0405
Equalized Odds diff       0.0806      0.0917

--- GhEduData - Location (treatment 2 vs 0) ---
                        Baseline  Engineered
CI width                  0.3514      0.3036
AUC-PR                    0.5534      0.5555
Macro F1                  0.6118      0.6791
At-risk recall            0.3200      0.5400
Equal Opportunity diff   -0.1750      0.1000
Equalized Odds diff       0.1750      0.2219



In [26]:
# 6c. School type: baseline vs engineered
X_train, X_test, T_train, T_test, Y_train, Y_test = train_test_split(
    X_gh_schooltype_trimmed, T_gh_schooltype_trimmed, Y_gh_schooltype_trimmed,
    test_size=0.2, random_state=42, stratify=Y_gh_schooltype_trimmed
)

run_comparison(X_train, T_train, Y_train, X_test, T_test, Y_test,
               gh_quarter_weight, label="GhEduData - School Type");


--- GhEduData - School Type (treatment 1 vs 0) ---
                        Baseline  Engineered
CI width                  0.3140      0.2514
AUC-PR                    0.5900      0.5905
Macro F1                  0.6212      0.7408
At-risk recall            0.2500      0.5500
Equal Opportunity diff   -0.2703     -0.2342
Equalized Odds diff       0.2703      0.2342

